# 27_YOLO 모델 학습하기

## 학습목표 
- 1. 변형한 데이터를 활용하여 YOLO 훈련을 진행하고, 훈련 시 사용 가능한 옵션에 대해 이해합니다.
  2. 훈련을 위한 폴더 구조와 yaml 파일을 만드는 방식을 이해합니다.
  3. 옵션인 comet_ml을 이용하여 훈련의 과정을 기록(로깅)합니다.

In [21]:
#훈련하기 위한 라이브러리 세팅
#!pip install ultralytics

### 1. 폴더 구조를 구성합니다.

In [ ]:
#이미지 파일로 폴더 구조를 알아봅시다.

In [5]:
#shutil을 사용해서 각 리스트 안에 있는 데이터들을 목적 폴더로 옮김
train_image_list = 'C:/Users/jeong/Desktop/최종코드/Datasets/Proj_1/train_image_list.txt'
valid_image_list = 'C:/Users/jeong/Desktop/최종코드/Datasets/Proj_1/valid_image_list.txt'

with open(train_image_list, "r", encoding="utf-8") as f:
    train_images = [line.strip() for line in f]  # 개행 문자 제거 후 리스트로 변환

with open(valid_image_list, "r", encoding="utf-8") as f:
    valid_images = [line.strip() for line in f]  # 개행 문자 제거 후 리스트로 변환

In [6]:
train_images[:3]

['C:/Users/jeong/Desktop/최종코드/Datasets/Proj_1/Images/upper_29244.png',
 'C:/Users/jeong/Desktop/최종코드/Datasets/Proj_1/Images/upper_29249.png',
 'C:/Users/jeong/Desktop/최종코드/Datasets/Proj_1/Images/upper_29229.png']

In [7]:
valid_images[:3]

['C:/Users/jeong/Desktop/최종코드/Datasets/Proj_1/Images/upper_29224.png',
 'C:/Users/jeong/Desktop/최종코드/Datasets/Proj_1/Images/upper_29227.png',
 'C:/Users/jeong/Desktop/최종코드/Datasets/Proj_1/Images/upper_29236.png']

In [9]:
file_name = train_images[0].split('/')[-1]
print(file_name)

upper_29244.png


In [ ]:
import os
import shutil
import ultralytics

In [13]:
Target = 'C:/Users/jeong/Desktop/최종코드/Datasets/Proj_1/YOLO/Train/images/'
for i in range(len(train_images)):
    file_name = train_images[i].split('/')[-1]
    origin = os.path.join(Target, file_name)
    #print(f'{train_images[i]} to {origin}')
    shutil.copy(train_images[i], origin)

In [14]:
Target = 'C:/Users/jeong/Desktop/최종코드/Datasets/Proj_1/YOLO/Valid/images/'
for i in range(len(valid_images)):
    file_name = valid_images[i].split('/')[-1]
    origin = os.path.join(Target, file_name)
    #print(f'{train_images[i]} to {origin}')
    shutil.copy(valid_images[i], origin)

### 2. YAML 파일을 작성합니다.

#### 특정 데이터에 대한 data Yaml 파일을 작성

In [ ]:
# Ultralytics 🚀 AGPL-3.0 License - https://ultralytics.com/license

# COCO8 dataset (first 8 images from COCO train2017) by Ultralytics
# Documentation: https://docs.ultralytics.com/datasets/detect/coco8/
# Example usage: yolo train data=coco8.yaml
# parent
# ├── ultralytics
# └── datasets
#     └── coco8  ← downloads here (1 MB)

# Train/val/test sets as 1) dir: path/to/imgs, 2) file: path/to/imgs.txt, or 3) list: [path/to/imgs1, path/to/imgs2, ..]
path: ../datasets/coco8 # dataset root dir
train: 이미지파일에 대한 목록.txt # train images (relative to 'path') 4 images
val: 이미지파일에 대한 목록.txt # val images (relative to 'path') 4 images
test: # test images (optional)

# Classes
names:
  0: person

## 3. 훈련의 옵션을 알아봅니다.        
- comet_ml을 활용하여 로깅할 수 있도록 추가 셋업합니다.

In [17]:
#!pip install comet_ml

In [18]:
import comet_ml
comet_ml.init()

COMET WARNING: comet_ml.init() is deprecated and will be removed soon. Please use comet_ml.login()


Please paste your Comet API key from https://www.comet.com/api/my/settings/
(api key may not show as you type)


Comet API key:  ········


COMET INFO: Valid Comet API Key saved in C:\Users\jeong\.comet.config (set COMET_CONFIG to change where it is saved).


In [ ]:
from ultralytics import YOLO

#Load a model
# 1.yaml 파일을 활용하여 모델을 세팅하는 경우
model = YOLO("yolo11n.yaml")  # build a new model from YAML

# 2.사전훈련된 모델의 가중치를 가져오는 경우
# model = YOLO("yolo11n.pt")  # load a pretrained model (recommended for training)

# 3.yaml파일로 세팅을 완료한 후, 사전훈련된 가중치를 덮어씌워줌
#model = YOLO("yolo11n.yaml").load("yolo11n.pt")  

#Train the model
#epochs = 훈련하려는 에포크 수 
#imgsz = 이미지의 사이즈
#device = GPU가 설정되어 있는 경우, device=0
results = model.train(data="C:/Users/jeong/Desktop/최종코드/Datasets/Proj_1/data_config.yaml", epochs=10, imgsz=640, device=0)

In [ ]:
from ultralytics import YOLO
from comet_ml import Experiment

# ✅ Comet ML 실험 생성
experiment = Experiment(
    api_key="s3KAHUfAQ5TDBCpJoffdZx9Dk",  # Comet API 키 입력
    project_name="yolov8",  # 프로젝트 이름 지정
    workspace="yolov8_1"  # 워크스페이스 이름 (Comet에서 확인 가능)
)

#Load a model
# 1.yaml 파일을 활용하여 모델을 세팅하는 경우
model = YOLO("yolo11n.yaml")  # build a new model from YAML

#Train the model
#epochs = 훈련하려는 에포크 수 
#imgsz = 이미지의 사이즈
#device = GPU가 설정되어 있는 경우, device=0
results = model.train(data="C:/Users/jeong/Desktop/최종코드/Datasets/Proj_1/data_config.yaml", epochs=10, imgsz=640, device=0)